<a href="https://colab.research.google.com/github/ArkanUbaidillah/Arkan-Ubaidillah-Warman_2411537001_ML2526/blob/main/Praktikum5/TugasCrossValidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
train_url = "https://raw.githubusercontent.com/ArkanUbaidillah/Arkan-Ubaidillah-Warman_2411537001_ML2526/main/Praktikum5/train.csv"
test_url = "https://raw.githubusercontent.com/ArkanUbaidillah/Arkan-Ubaidillah-Warman_2411537001_ML2526/main/Praktikum5/test.csv"
submit_url = "https://raw.githubusercontent.com/ArkanUbaidillah/Arkan-Ubaidillah-Warman_2411537001_ML2526/main/Praktikum5/gender_submission.csv"

train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)
submit_df = pd.read_csv(submit_url)

In [3]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

train = train_df[['Survived'] + features].copy()
test = test_df[features].copy()

In [4]:
imputer = SimpleImputer(strategy='mean')

train['Age'] = imputer.fit_transform(train[['Age']])
test['Age'] = imputer.transform(test[['Age']])

train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
test['Embarked'] = test['Embarked'].fillna(train['Embarked'].mode()[0])

In [5]:
le = LabelEncoder()

train['Sex'] = le.fit_transform(train['Sex'])
test['Sex'] = le.transform(test['Sex'])

train['Embarked'] = le.fit_transform(train['Embarked'])
test['Embarked'] = le.transform(test['Embarked'])

In [6]:
X = train.drop('Survived', axis=1)
y = train['Survived']

In [7]:
model = LogisticRegression(max_iter=1000)

In [8]:
def evaluate_model(k):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    acc = cross_val_score(model, X, y, cv=kf, scoring='accuracy')

    print(f"\n=== K = {k} ===")
    print("Accuracy :", np.mean(acc))

for k in [5, 7, 10]:
    evaluate_model(k)


=== K = 5 ===
Accuracy : 0.7990898248697508

=== K = 7 ===
Accuracy : 0.7934476940382452

=== K = 10 ===
Accuracy : 0.7945568039950063


In [10]:
model.fit(X, y)

# Impute missing 'Fare' values in the 'test' DataFrame
# using the mean of 'Fare' from the 'train' DataFrame.
if 'Fare' in test.columns and test['Fare'].isnull().any():
    fare_mean_from_train = train['Fare'].mean()
    test['Fare'] = test['Fare'].fillna(fare_mean_from_train)

y_pred = model.predict(test)

In [11]:
y_true = submit_df['Survived']

print("\n=== Evaluasi pada Test Data ===")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1 Score :", f1_score(y_true, y_pred))


=== Evaluasi pada Test Data ===
Accuracy : 0.9425837320574163
Precision: 0.9155844155844156
Recall   : 0.9276315789473685
F1 Score : 0.9215686274509803
